# Credit Risk Modeling - Full Data Science Project

End-to-end credit risk classification project using the **German Credit Risk** dataset
(Kaggle: `uciml/german-credit`). The workflow moves through data analysis and
visualization, feature engineering, model training/tuning, and a Streamlit
deployment app for serving predictions.

**Pipeline:**
1. Data Loading & Cleaning
2. Exploratory Data Analysis (univariate, bivariate, correlation)
3. Risk-Focused Analysis
4. Feature Engineering & Encoding
5. Train/Test Split
6. Model Training & Hyperparameter Tuning (Decision Tree, Random Forest, Extra Trees, XGBoost)
7. Model Export
8. Streamlit App (`app.py`)


## 1. Setup & Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')


## 2. Load the Dataset

Expects `german_credit_data.csv` in the same directory, with columns:
`index, Age, Sex, Job, Housing, Saving accounts, Checking account, Credit amount, Duration, Purpose, Risk`.


In [ ]:
df = pd.read_csv('german_credit_data.csv')
df.head()


In [ ]:
df.shape


In [ ]:
df.info()


## 3. Initial Cleaning

- Drop the redundant index column (pandas already provides one).
- Inspect and handle missing values in `Saving accounts` and `Checking account`.
- Check for duplicate rows.


In [ ]:
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

df.columns = df.columns.str.strip()
df.columns


In [ ]:
df.isna().sum()


In [ ]:
df.duplicated().sum()


In [ ]:
# Rows with no recorded savings/checking account status are dropped rather than
# imputed, since "no account" is materially different from any observed category.
df = df.dropna().reset_index(drop=True)
df.shape


In [ ]:
df.describe(include='all').T


## 4. Univariate Analysis

### 4.1 Numeric feature distributions


In [ ]:
numeric_cols = ['Age', 'Credit amount', 'Duration']

df[numeric_cols].hist(bins=15, edgecolor='black', figsize=(12, 4))
plt.suptitle('Distribution of Numerical Features', fontsize=14)
plt.tight_layout()
plt.show()


### 4.2 Boxplots — outlier inspection

In [ ]:
plt.figure(figsize=(12, 5))
for i, col in enumerate(numeric_cols):
    plt.subplot(1, 3, i + 1)
    sns.boxplot(y=df[col], color='skyblue')
    plt.title(col)
plt.tight_layout()
plt.show()


In [ ]:
# Inspect long-duration outliers
df.query('Duration > 60')[['Duration', 'Purpose']]


### 4.3 Categorical feature distributions

In [ ]:
categorical_cols = ['Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']

plt.figure(figsize=(15, 10))
for i, col in enumerate(categorical_cols):
    plt.subplot(3, 2, i + 1)
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order)
    plt.title(f'Distribution of {col}')
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 5. Bivariate Analysis & Correlations

In [ ]:
corr = df[['Age', 'Job', 'Credit amount', 'Duration']].corr()

plt.figure(figsize=(6, 5))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()


In [ ]:
df.groupby('Job')['Credit amount'].mean()


In [ ]:
df.groupby('Sex')['Credit amount'].mean()


In [ ]:
pd.pivot_table(df, values='Credit amount', index='Housing', columns='Purpose', aggfunc='mean')


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df, x='Age', y='Credit amount',
    hue='Sex', size='Duration', alpha=0.7, palette='Set1'
)
plt.title('Credit Amount vs Age, colored by Sex, sized by Duration')
plt.show()


In [ ]:
plt.figure(figsize=(8, 6))
sns.violinplot(data=df, x='Saving accounts', y='Credit amount', palette='pastel')
plt.title('Credit Amount Distribution by Saving Accounts')
plt.show()


## 6. Risk-Focused Analysis

In [ ]:
df['Risk'].value_counts(normalize=True) * 100


In [ ]:
plt.figure(figsize=(12, 4))
for i, col in enumerate(numeric_cols):
    plt.subplot(1, 3, i + 1)
    sns.boxplot(data=df, x='Risk', y=col, palette='pastel')
    plt.title(f'{col} by Risk')
plt.tight_layout()
plt.show()


In [ ]:
df.groupby('Risk')[['Age', 'Credit amount', 'Duration']].mean()


In [ ]:
plt.figure(figsize=(10, 10))
for i, col in enumerate(categorical_cols):
    plt.subplot(3, 2, i + 1)
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, hue='Risk', order=order, palette='Set1')
    plt.title(f'{col} by Risk')
    plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


## 7. Feature Engineering

Select the modeling features, then label-encode categorical columns and the
target. Encoders are persisted with `joblib` so the Streamlit app can apply
identical transformations at inference time.


In [ ]:
features = ['Age', 'Sex', 'Job', 'Housing', 'Saving accounts', 'Checking account', 'Credit amount', 'Duration']
target = 'Risk'

df_model = df[features + [target]].copy()
df_model.head()


In [ ]:
from sklearn.preprocessing import LabelEncoder
import joblib

categorical_cols_model = df_model.select_dtypes(include='object').columns.drop(target)
label_encoders = {}

for col in categorical_cols_model:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col])
    label_encoders[col] = le
    joblib.dump(le, f'{col}_encoder.pickle')

categorical_cols_model


In [ ]:
target_encoder = LabelEncoder()
df_model[target] = target_encoder.fit_transform(df_model[target])
joblib.dump(target_encoder, 'target_encoder.pickle')

# Confirm mapping: which label is 'good' (low risk) vs 'bad' (high risk)
dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))


In [ ]:
df_model[target].value_counts()


In [ ]:
df_model.head()


## 8. Train/Test Split

No feature scaling is applied since every model considered here is tree-based
and scale-invariant.


In [ ]:
from sklearn.model_selection import train_test_split

X = df_model.drop(columns=[target])
y = df_model[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=1
)

X_train.shape, X_test.shape


## 9. Model Training & Hyperparameter Tuning

A shared helper runs `GridSearchCV` (5-fold, accuracy-scored) for any given
model/parameter grid and returns the tuned estimator, its test accuracy, and
the chosen parameters.


In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV


def train_model(model, param_grid, X_train, y_train, X_test, y_test):
    grid = GridSearchCV(model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
    grid.fit(X_train, y_train)

    best_model = grid.best_estimator_
    y_pred = best_model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)

    return best_model, accuracy, grid.best_params_


### 9.1 Decision Tree

In [ ]:
dt = DecisionTreeClassifier(random_state=1, class_weight='balanced')

dt_param_grid = {
    'max_depth': [3, 5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

best_dt, acc_dt, params_dt = train_model(dt, dt_param_grid, X_train, y_train, X_test, y_test)
print(f'Decision Tree accuracy: {acc_dt:.2f}')
print('Best params:', params_dt)


### 9.2 Random Forest

In [ ]:
rf = RandomForestClassifier(random_state=1, class_weight='balanced', n_jobs=-1)

rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [5, 7, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
}

best_rf, acc_rf, params_rf = train_model(rf, rf_param_grid, X_train, y_train, X_test, y_test)
print(f'Random Forest accuracy: {acc_rf:.2f}')
print('Best params:', params_rf)


### 9.3 Extra Trees

In [ ]:
et = ExtraTreesClassifier(random_state=1, class_weight='balanced', n_jobs=-1)

et_param_grid = rf_param_grid  # same search space as Random Forest

best_et, acc_et, params_et = train_model(et, et_param_grid, X_train, y_train, X_test, y_test)
print(f'Extra Trees accuracy: {acc_et:.2f}')
print('Best params:', params_et)


### 9.4 XGBoost

In [ ]:
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb = XGBClassifier(
    random_state=1,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric='logloss',
)

xgb_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.1, 0.2],
    'subsample': [0.7, 1.0],
    'colsample_bytree': [0.7, 1.0],
}

best_xgb, acc_xgb, params_xgb = train_model(xgb, xgb_param_grid, X_train, y_train, X_test, y_test)
print(f'XGBoost accuracy: {acc_xgb:.2f}')
print('Best params:', params_xgb)


## 10. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model': ['Decision Tree', 'Random Forest', 'Extra Trees', 'XGBoost'],
    'Accuracy': [acc_dt, acc_rf, acc_et, acc_xgb],
})
results.sort_values('Accuracy', ascending=False)


In [ ]:
best_model_name = results.sort_values('Accuracy', ascending=False).iloc[0]['Model']
model_lookup = {
    'Decision Tree': best_dt,
    'Random Forest': best_rf,
    'Extra Trees': best_et,
    'XGBoost': best_xgb,
}
best_model = model_lookup[best_model_name]

y_pred_best = best_model.predict(X_test)
print(f'Best model: {best_model_name}')
print(classification_report(y_test, y_pred_best, target_names=target_encoder.classes_))


In [ ]:
sns.heatmap(
    confusion_matrix(y_test, y_pred_best), annot=True, fmt='d', cmap='Blues',
    xticklabels=target_encoder.classes_, yticklabels=target_encoder.classes_,
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title(f'Confusion Matrix — {best_model_name}')
plt.show()


## 11. Export the Best Model

Persist the winning estimator alongside the encoders exported in Step 7 so
the Streamlit app can load everything it needs at inference time.


In [ ]:
joblib.dump(best_model, 'credit_risk_model.pickle')
print(f'Exported {best_model_name} to credit_risk_model.pickle')


## 12. Streamlit Deployment App

See `app.py` in this directory. Run it with:

```bash
streamlit run app.py
```

